# 🌿 Crop Disease Detection — GPU Training Pipeline
### Approved 4-Crop Architecture Remaining Training Jobs:
1. **Grape Unified Model** (G1 Niphad + G2 2024 — 7 Classes, 6,203 images)
2. **Chilli Model** (C1 COLD 2024 — 5 Classes, 1,932 images)
3. **Sugarcane Unified Model** (S1 Maharashtra + S2 Large — 11 Classes, 8,926 images)

*(Note: T1 Tomato baseline is already completed at 90.23% accuracy and remains untouched.)*

---
### Instructions for Execution:
- **On Kaggle Notebooks:** Under `Notebook options` (right sidebar), set `Accelerator` to `GPU T4 x2` (or `GPU T4`).
- **On Google Colab:** Click `Runtime` -> `Change runtime type` -> select `T4 GPU` -> `Save`.
- Total expected runtime on a T4 GPU: **~20 to 25 minutes**.

In [ ]:
# 1. Hardware & Environment Verification
import tensorflow as tf
import subprocess

print(f'TensorFlow Version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow GPU Available: {len(gpus) > 0}')
if gpus:
    try:
        gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
        print(f'GPU Detected: {gpu_name}')
    except Exception:
        for g in gpus:
            print(f'GPU Device: {g.name}')
else:
    print('⚠️ WARNING: No GPU detected! Please enable GPU accelerator in notebook settings.')

In [ ]:
# 2. Clone Repository or Set Project Path
import os

# If running directly in a cloned repo on Colab/Kaggle:
if not os.path.exists('train_experiment.py'):
    !git clone https://github.com/sohail-148/maharashtra-crop-disease-detection.git project
    %cd project

print('Current Working Directory:', os.getcwd())
!ls -lh

In [ ]:
# 3. Configure Dataset Root
import os

# Explicit Kaggle mount path for sohail1148/crop-disease-datasets:
KAGGLE_PATH = '/kaggle/input/datasets/sohail1148/crop-disease-datasets'

if os.path.exists(KAGGLE_PATH):
    DATASET_ROOT = KAGGLE_PATH
elif os.path.exists('/kaggle/input/crop-disease-datasets'):
    DATASET_ROOT = '/kaggle/input/crop-disease-datasets'
else:
    # Fallback to local project directory
    DATASET_ROOT = os.getcwd()

print(f'Using DATASET_ROOT: {DATASET_ROOT}')
!python train_experiment.py --help

In [ ]:
# 4. Pre-Training Validation & Integrity Check
import os
import glob
import pandas as pd
from train_experiment import rewrite_paths

print('=== PRE-TRAINING VALIDATION ===')

# A. Confirm the five dataset folders and exact image counts
expected_counts = {
    'grape_niphad': 2726,
    'grape_2024': 3477,
    'chilli_cold': 1932,
    'sugarcane_maharashtra': 2521,
    'sugarcane_large': 6405,
}

valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_folders_ok = True

print('\n1. Checking 5 Source Dataset Folders:')
for folder, expected in expected_counts.items():
    folder_path = os.path.join(DATASET_ROOT, folder)
    if not os.path.isdir(folder_path):
        print(f'  ❌ MISSING: {folder_path}')
        all_folders_ok = False
        continue
    files = [f for f in glob.glob(f'{folder_path}/**/*', recursive=True)
             if os.path.splitext(f)[1].lower() in valid_exts and 'desktop.ini' not in f.lower()]
    count_ok = (len(files) == expected)
    status = '✅ PASS' if count_ok else f'❌ FAIL (found {len(files)}, expected {expected})'
    print(f'  - {folder:22s}: {len(files):>5} images (expected {expected}) -> {status}')
    if not count_ok:
        all_folders_ok = False

assert all_folders_ok, 'Source dataset verification failed!'

# B. Confirm the three training split directories exist
print('\n2. Checking 3 Training Split Directories:')
split_dirs = [
    'splits/grape_unified',
    'splits/chilli_cold',
    'splits/sugarcane_unified',
]

for s_dir in split_dirs:
    assert os.path.isdir(s_dir), f'Missing split directory: {s_dir}'
    for required_file in ['class_index.csv', 'train.csv', 'val.csv', 'test.csv']:
        f_path = os.path.join(s_dir, required_file)
        assert os.path.isfile(f_path), f'Missing file: {f_path}'
    print(f'  - {s_dir:25s}: ✅ Verified (class_index, train, val, test)')

# C. Verify sample CSV paths after rewrite_paths() exist under Kaggle dataset root
print('\n3. Verifying Sample Rewritten Paths:')
for exp, s_dir in [('GRAPE', 'splits/grape_unified'), ('CHILLI', 'splits/chilli_cold'), ('SUGARCANE', 'splits/sugarcane_unified')]:
    tr = pd.read_csv(os.path.join(s_dir, 'train.csv'))
    rewritten = rewrite_paths(tr, DATASET_ROOT)
    sample_paths = rewritten['file_path'].head(3).tolist()
    print(f'  [{exp}] First rewritten path: {sample_paths[0]}')
    for p in sample_paths:
        assert os.path.exists(p), f'Rewritten file does not exist: {p}'
    print(f'  [{exp}] Sample paths verified under DATASET_ROOT.')

print('\n=== PRE-TRAINING VALIDATION RESULT: ALL CHECKS PASSED ✅ ===')

In [ ]:
# 5. Train Job 1 — Grape Unified Model (7 Classes)
# Combines G1 Niphad (Maharashtra) + G2 2024 into a single robust model
!python train_experiment.py --experiment GRAPE --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 6. Train Job 2 — Chilli Model (5 Classes)
# Trains C1 COLD 2024 dataset
!python train_experiment.py --experiment CHILLI --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 7. Train Job 3 — Sugarcane Unified Model (11 Classes)
# Combines S1 Maharashtra + S2 Large into a single comprehensive model
!python train_experiment.py --experiment SUGARCANE --dataset-root "$DATASET_ROOT" --batch-size 64

In [ ]:
# 8. Display Metrics & Final Evaluation Summary
import pandas as pd
import glob

results = []
for m in ['grape_unified', 'chilli_cold', 'sugarcane_unified']:
    p = f'results/{m}/test_metrics.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        results.append(df)

if results:
    summary_df = pd.concat(results, ignore_index=True)
    print('=== ALL 3 TRAINED MODELS FINAL TEST RESULTS ===')
    display(summary_df)
else:
    print('Results not generated yet.')

In [ ]:
# 9. Package Trained Models & Results for 1-Click Download
import shutil
!zip -r trained_models_results.zip models/ results/
print('Archive created: trained_models_results.zip')

# If running in Google Colab, trigger direct browser download:
try:
    from google.colab import files
    files.download('trained_models_results.zip')
except Exception:
    print('On Kaggle: download trained_models_results.zip from the right-hand Output panel.')